# G1 P0・P1・P2：1.9 m/s 転倒過程の骨格比較（Isaac Gym）

**目的：** 高速歩行評価と同じ条件（指令速度1.8，1.9 m/s×各8試行，シード2026，16環境）で各方策を18秒間再実行し，**1.9 m/sの試行1（rep=1）**の剛体リンク3次元位置を保存する．

- P0，P1，P2の学習済み`model_6700.pt`を読み取り専用で使用し，再学習しない．
- 同条件の高速評価では，P1のrep=1が約3.74秒で終了した．終了時刻は再実行で照合する．
- **重要：エピソード終了（done）は接触・ロール・ピッチなどによる終了判定であり，必ずしも身体が完全に床に倒れたことを意味しない．**
- Isaac Gymのヘッドレス動作で，カメラによるレンダリングは行わない．実際の剛体リンク中心の3D座標を記録し，CPUで模式骨格を描画する．
- 終了直前の姿勢を，`reset_idx`の前にフックで保存する．終了後はその最終姿勢を静止表示し，P0・P2の動作は18秒まで表示する．
- 元の1.4 m/s動画フォルダー`video_1p4_compare`と既存の高速評価`fine_1p9_20261028`は変更しない．
- **実行：ColabのT4 GPUを選び，上から全セルを実行する．**


## 1．Google Drive／GPUを準備

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, os, json, shutil, sys, time
from datetime import datetime

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/Colab Notebooks/PhysicalAI応用編１/G1_RL_Training')
assert DRIVE.is_dir(), DRIVE
EXP = DRIVE / 'contact_tolerance_P0_P1_P2_from6500_20261024_075342'
assert EXP.is_dir(), f'先日のP0/P1/P2学習フォルダーが見つかりません: {EXP}'
OUT_DIR = EXP / 'evaluation' / 'high_speed_limit_P012_coarse_20261024'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('元の高速歩行評価は変更せず，動画用の新しいフォルダーに保存します．')
print('新規出力:', OUT_DIR)
subprocess.run(['nvidia-smi'],check=True)
VIDEO_DIR = OUT_DIR / 'video_1p9_fall_rep1'
VIDEO_DIR.mkdir(parents=True, exist_ok=True)
print('動画保存先:', VIDEO_DIR)


## 2．Isaac Gymと学習済み方策を準備（既存ノートブックと同じ設定）

In [ ]:
WORK = Path('/content')
repo = WORK / 'unitree_rl_gym'
rsl = WORK / 'rsl_rl'
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/unitreerobotics/unitree_rl_gym.git',str(repo)],check=True)
if not rsl.exists():
    subprocess.run(['git','clone','https://github.com/leggedrobotics/rsl_rl.git',str(rsl)],check=True)
subprocess.run(['git','-C',str(repo),'checkout','276801e46c5d433564f24658bac64f254b7d2d4b'],check=True)
subprocess.run(['git','-C',str(rsl),'checkout','2ad79cf0caa85b91721abfe358105f869a784121'],check=True)
print('unitree_rl_gym',subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip())
print('rsl_rl',subprocess.check_output(['git','-C',str(rsl),'rev-parse','HEAD'],text=True).strip())

In [ ]:
ARCHIVE = DRIVE / 'IsaacGym_Preview_4_Package.tar'
print('期待するファイル:', ARCHIVE)
if not ARCHIVE.is_file():
    print('ファイル未配置です．公式配布サイトから入手したtarファイルをGoogle Driveへアップロードしてください．')
else:
    print('検出:', ARCHIVE.name, '容量 GiB:', round(ARCHIVE.stat().st_size / 1024**3, 2))

In [ ]:
from pathlib import Path
import subprocess, os

MAMBA_BIN = Path('/content/bin/micromamba')
if not MAMBA_BIN.exists():
    subprocess.run(['bash','-lc',
      'mkdir -p /content && curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba'], check=True)
PY38 = Path('/content/g1-python38/bin/python')
env = os.environ.copy()
env['MAMBA_ROOT_PREFIX'] = '/content/mamba-root'
if not PY38.exists():
    subprocess.run([str(MAMBA_BIN), 'create','-y','-p','/content/g1-python38',
         '-c','conda-forge','python=3.8','pip'], check=True, env=env)
print('Python 3.8:', subprocess.check_output([str(PY38), '--version'], text=True).strip())

In [ ]:
from pathlib import Path
import subprocess, os

assert ARCHIVE.is_file(), f'Isaac Gym配布物がありません: {ARCHIVE}'
assert PY38.exists(), '先にPython 3.8環境を作成してください．'

EXTRACT = Path('/content/isaacgym_dist')
if not EXTRACT.exists():
    EXTRACT.mkdir(parents=True)
    subprocess.run(['tar','-xf',str(ARCHIVE),'-C',str(EXTRACT)], check=True)

isaac_python_dirs = [p for p in EXTRACT.rglob('python') if (p/'setup.py').is_file() and p.parent.name == 'isaacgym']
assert isaac_python_dirs, '配布物から isaacgym/python/setup.py が見つかりません．構成を確認してください．'
ISAAC_PY_DIR = isaac_python_dirs[0]
print('Isaac Gym Python package:', ISAAC_PY_DIR)

def pip(*args):
    subprocess.run([str(PY38),'-m','pip',*map(str,args)],check=True)

pip('install','--upgrade','pip','setuptools','wheel')
pip('install','numpy==1.23.5','gym==0.23.1','imageio','ninja','tensorboard')
pip('install','torch==2.3.1','torchvision==0.18.1','torchaudio==2.3.1',
    '--index-url','https://download.pytorch.org/whl/cu121')
# 既存GPU版PyTorchを維持し，旧Isaac Gymインストールの依存関係で置き換えない
pip('install','-e',str(ISAAC_PY_DIR),'--no-deps')
pip('install','-e',str(rsl))
pip('install','-e',str(repo))
print('依存ライブラリのインストールが終了しました．まだGPU動作の確認は済んでいません．')

In [ ]:
PY38 = Path('/content/g1-python38/bin/python')
assert PY38.is_file(), PY38
RUN_ENV = os.environ.copy()
RUN_ENV['LD_LIBRARY_PATH'] = '/content/g1-python38/lib:' + RUN_ENV.get('LD_LIBRARY_PATH','')
RUN_ENV['TORCH_EXTENSIONS_DIR'] = '/content/g1_torch_extensions'
RUN_ENV['PATH'] = str(PY38.parent) + os.pathsep + RUN_ENV.get('PATH','')
Path(RUN_ENV['TORCH_EXTENSIONS_DIR']).mkdir(exist_ok=True)
subprocess.run([str(PY38),'-m','pip','install','-q','ninja'],check=True,env=RUN_ENV)
p=subprocess.run([str(PY38),'-c',
 'from isaacgym import gymapi; import torch; from torch.utils.cpp_extension import is_ninja_available; '
 'print("CUDA:",torch.cuda.is_available(),"Ninja:",is_ninja_available())'],
 cwd=str(repo),env=RUN_ENV,text=True,capture_output=True)
print(p.stdout,p.stderr[-2000:]); assert p.returncode==0
assert 'CUDA: True Ninja: True' in p.stdout

In [ ]:
MODELS={key:EXP/key/'checkpoints'/'model_6700.pt' for key in ('P0','P1','P2')}
for name,f in MODELS.items():
    assert f.is_file() and f.stat().st_size>100000, f
    print(name, f, round(f.stat().st_size/1024),'KiB')
print('学習済みファイルは読み取り専用で利用します．')

## 3．18秒間の物理シミュレーションと，終了直前の剛体姿勢記録スクリプト

In [ ]:
from pathlib import Path
import py_compile
SIM_SCRIPT = Path('/content/g1_p012_1p9_fall_capture.py')
SIM_SCRIPT.write_text("import argparse, csv, json, hashlib, sys\nfrom pathlib import Path\nimport numpy as np\nfrom isaacgym import gymapi, gymtorch  # must be before importing torch\nimport torch\nfrom legged_gym.envs import *\nfrom legged_gym.utils import task_registry, get_args\n\nap=argparse.ArgumentParser(add_help=False)\nap.add_argument('--checkpoint', required=True)\nap.add_argument('--label', required=True)\nap.add_argument('--out', required=True)\nap.add_argument('--experiment-config', required=True)\nopt, other=ap.parse_known_args()\nsys.argv=[sys.argv[0]]+other\nargs=get_args()\nassert args.task=='g1'\nex=json.loads(Path(opt.experiment_config).read_text(encoding='utf-8'))\nSPEEDS=tuple(float(v) for v in ex['speeds_mps'])\nREPS=int(ex['reps'])\nDURATION_S=float(ex['duration_s'])\nSEED=int(ex['seed'])\nSAMPLE_EVERY=int(ex.get('sample_every_steps',5))\nN=len(SPEEDS)*REPS\nargs.seed=SEED\nargs.num_envs=N\n\ncfg,train_cfg=task_registry.get_cfgs(name='g1')\ncfg.env.num_envs=N\ncfg.seed=SEED\ncfg.env.test=False\ncfg.noise.add_noise=False\ncfg.domain_rand.randomize_friction=False\ncfg.domain_rand.randomize_base_mass=False\ncfg.domain_rand.push_robots=False\ncfg.terrain.mesh_type='plane'\ncfg.terrain.curriculum=False\ncfg.commands.curriculum=False\ncfg.commands.heading_command=False\ncfg.commands.resampling_time=1_000_000.0\n\nenv,_=task_registry.make_env(name='g1',args=args,env_cfg=cfg)\nrunner,_=task_registry.make_alg_runner(env=env,name='g1',args=args,\n                                       train_cfg=train_cfg,log_root=None)\nckpt=torch.load(opt.checkpoint,map_location=env.device)\nassert 'model_state_dict' in ckpt\nrunner.alg.actor_critic.load_state_dict(ckpt['model_state_dict'])\npolicy=runner.get_inference_policy(device=env.device)\nassert runner.alg.actor_critic.memory_a.hidden_states is None\n\ndt=float(env.dt)\nSTEPS=int(round(DURATION_S/dt))\nassert abs(STEPS*dt-DURATION_S)<1e-3\nassert STEPS<env.max_episode_length, 'episode timeout would confound fall check'\nassert env.num_envs==N\nprint('Evaluation settings:',opt.label,'envs',N,'steps',STEPS,'dt',dt,flush=True)\n\n# Each policy uses precisely the same random initial states.\ntorch.manual_seed(SEED)\ntorch.cuda.manual_seed_all(SEED)\nperturb=(torch.rand((N,env.num_dof),device=env.device)-0.5)*0.04\nenv.dof_pos[:]=env.default_dof_pos+perturb\nenv.dof_vel[:]=0.0\nenv.root_states[:,7:13]=0.0\nenv.gym.set_dof_state_tensor(env.sim,gymtorch.unwrap_tensor(env.dof_state))\nenv.gym.set_actor_root_state_tensor(env.sim,gymtorch.unwrap_tensor(env.root_states))\nenv.base_lin_vel[:]=0.0\nenv.base_ang_vel[:]=0.0\nenv.episode_length_buf[:]=0\nenv.actions[:]=0.0\nenv.last_actions[:]=0.0\n\ntargets=torch.tensor([v for v in SPEEDS for _ in range(REPS)],device=env.device,dtype=torch.float32)\nenv.commands[:]=0.0\nenv.commands[:,0]=targets\nif hasattr(env,'phase'):env.phase[:]=0.0\nif hasattr(env,'phase_left'):env.phase_left[:]=0.0\nif hasattr(env,'phase_right'):env.phase_right[:]=0.5\nif hasattr(env,'leg_phase'):\n    env.leg_phase[:,0]=0.0\n    env.leg_phase[:,1]=0.5\nenv.compute_observations()\nobs=env.get_observations()\nassert obs.shape==(N,47)\nassert torch.allclose(env.commands[:,0],targets)\n\nroot=env.root_states.detach().cpu().numpy().copy()\nq=env.dof_pos.detach().cpu().numpy().copy()\nqd=env.dof_vel.detach().cpu().numpy().copy()\ninitial_sha=hashlib.sha256(np.ascontiguousarray(np.concatenate([\n    root[:,:13].flatten(),q.flatten(),qd.flatten()]).astype('<f4')).tobytes()).hexdigest()\nprint('initial sha256:',initial_sha,flush=True)\n\nalive=torch.ones(N,dtype=torch.bool,device=env.device)\nfalls=torch.full((N,),float('nan'),device=env.device)\nmin_height=np.full(N,np.inf)\nmax_abs_roll=np.zeros(N)\nmax_abs_pitch=np.zeros(N)\nsteady_n=np.zeros(N,dtype=np.int64)\nsteady_body_sum=np.zeros(N)\nsteady_world_sum=np.zeros(N)\nsteady_body_sqerr=np.zeros(N)\nlast_body=np.full(N,np.nan)\nlast_world=np.full(N,np.nan)\nraw=[]\n\n# --- Read-only rigid-body poses for the selected 1.9 m/s falling trial ---\nCAPTURE_SPEED = 1.9\nCAPTURE_REP = 1\nassert CAPTURE_SPEED in SPEEDS and CAPTURE_REP < REPS\nCAPTURE_ENV = SPEEDS.index(CAPTURE_SPEED)*REPS + CAPTURE_REP\nCAPTURE_FPS = 25\nCAPTURE_STRIDE = round(1.0 / (dt*CAPTURE_FPS))\nassert abs(CAPTURE_STRIDE * dt * CAPTURE_FPS - 1.0) < 1e-4\nbody_names=list(env.gym.get_actor_rigid_body_names(env.envs[0],env.actor_handles[0]))\nbody_states=gymtorch.wrap_tensor(env.gym.acquire_rigid_body_state_tensor(env.sim)).view(N,-1,13)\nassert len(body_names)==body_states.shape[1]\nstart_root_xy = env.root_states[CAPTURE_ENV, :2].detach().cpu().numpy().copy()\ncapture_times=[]\ncapture_xyz=[]\ncapture_root=[]\ncapture_yaw=[]\ncapture_fallen=False\n\n# This pinned legged_robot.py calls reset_idx() after check_termination() in\n# post_physics_step(). Capture the terminal physics state before auto-reset.\nterminal_pose = {}\noriginal_reset_idx = env.reset_idx\ndef reset_idx_with_capture(env_ids):\n    if not terminal_pose and bool((env_ids == CAPTURE_ENV).any().item()):\n        env.gym.refresh_rigid_body_state_tensor(env.sim)\n        terminal_pose['body_xyz'] = body_states[CAPTURE_ENV, :, :3].detach().cpu().numpy().copy()\n        terminal_pose['root_xyz'] = env.root_states[CAPTURE_ENV, :3].detach().cpu().numpy().copy()\n        terminal_pose['yaw_rad'] = float(env.rpy[CAPTURE_ENV, 2].item())\n    return original_reset_idx(env_ids)\nenv.reset_idx = reset_idx_with_capture\n\nwith torch.inference_mode():\n  for step in range(STEPS):\n    t=round((step+1)*dt,5)\n    was_alive=alive.clone()\n    actions=policy(obs.detach())\n    obs,_,_,dones,_=env.step(actions.detach())\n    dones=dones.bool()\n    just_ended=was_alive & dones\n    falls[just_ended]=t\n    alive &= ~dones\n    runner.alg.actor_critic.memory_a.reset(dones)\n    # Save the terminal frame even when it falls between 25 fps samples.\n    if bool(just_ended[CAPTURE_ENV].item()):\n       assert terminal_pose, 'No pre-reset pose captured for terminal frame'\n       capture_xyz.append(terminal_pose['body_xyz'])\n       capture_root.append(terminal_pose['root_xyz'])\n       capture_yaw.append(terminal_pose['yaw_rad'])\n       capture_times.append(t)\n       capture_fallen=True\n    elif (step+1) % CAPTURE_STRIDE == 0 and bool(alive[CAPTURE_ENV].item()):\n       env.gym.refresh_rigid_body_state_tensor(env.sim)\n       capture_xyz.append(body_states[CAPTURE_ENV, :, 0:3].detach().cpu().numpy().copy())\n       capture_root.append(env.root_states[CAPTURE_ENV, :3].detach().cpu().numpy().copy())\n       capture_yaw.append(float(env.rpy[CAPTURE_ENV,2].item()))\n       capture_times.append(t)\n    good=(was_alive & ~dones).detach().cpu().numpy()\n\n    body_vx=env.base_lin_vel[:,0].detach().cpu().numpy()\n    world_vx=env.root_states[:,7].detach().cpu().numpy()\n    world_x=env.root_states[:,0].detach().cpu().numpy()\n    rpy=env.rpy[:,:3].detach().cpu().numpy()\n    height=env.base_pos[:,2].detach().cpu().numpy()\n\n    min_height[good]=np.minimum(min_height[good],height[good])\n    max_abs_roll[good]=np.maximum(max_abs_roll[good],np.abs(rpy[good,0]))\n    max_abs_pitch[good]=np.maximum(max_abs_pitch[good],np.abs(rpy[good,1]))\n    last_body[good]=body_vx[good]\n    last_world[good]=world_vx[good]\n    if t>=4.0:\n       idx=np.flatnonzero(good)\n       steady_n[idx]+=1\n       steady_body_sum[idx]+=body_vx[idx]\n       steady_world_sum[idx]+=world_vx[idx]\n       cmd=np.asarray(SPEEDS,dtype=float)[idx//REPS]\n       steady_body_sqerr[idx]+=(body_vx[idx]-cmd)**2\n\n    if (step+1)%SAMPLE_EVERY==0 or step==0:\n       for j in np.flatnonzero(good):\n          raw.append({'model':opt.label,'command_mps':SPEEDS[j//REPS],\n               'rep':j%REPS,'time_s':t,'body_vx_mps':float(body_vx[j]),\n               'world_vx_mps':float(world_vx[j]),'world_x_m':float(world_x[j]),\n               'roll_rad':float(rpy[j,0]),'pitch_rad':float(rpy[j,1]),\n               'yaw_rad':float(rpy[j,2]),'base_height_m':float(height[j])})\n    if not bool(alive.any().item()):\n       print('All environments terminated by step',step+1,flush=True)\n       break\n\nfall_np=falls.detach().cpu().numpy()\nrows=[]\nfor j in range(N):\n   v=SPEEDS[j//REPS]\n   survived=not np.isfinite(fall_np[j])\n   denom=max(1,steady_n[j])\n   mean_body=steady_body_sum[j]/denom if steady_n[j]>0 else float('nan')\n   mean_world=steady_world_sum[j]/denom if steady_n[j]>0 else float('nan')\n   rmse=np.sqrt(steady_body_sqerr[j]/denom) if steady_n[j]>0 else float('nan')\n   tolerance=max(.10,.15*v)\n   tracking=bool(survived and np.isfinite(mean_world) and np.isfinite(mean_body)\n                 and abs(mean_world-v)<=tolerance and abs(mean_body-v)<=tolerance)\n   rows.append({'model':opt.label,'command_mps':v,'rep':j%REPS,\n      'survived_18s':int(survived),\n      'termination_time_s':float(fall_np[j]) if not survived else '',\n      'steady_body_vx_mps':mean_body,'steady_world_vx_mps':mean_world,\n      'steady_body_rmse_mps':rmse,'speed_tolerance_mps':tolerance,\n      'survived_and_tracked':int(tracking),\n      'min_base_height_m':float(min_height[j]) if np.isfinite(min_height[j]) else '',\n      'max_abs_roll_rad':float(max_abs_roll[j]),\n      'max_abs_pitch_rad':float(max_abs_pitch[j]),\n      'observed_steady_samples':int(steady_n[j])})\n\nout=Path(opt.out)\nout.mkdir(parents=True,exist_ok=True)\ndef save_csv(filename,rs):\n   assert rs, filename\n   with (out/filename).open('w',newline='',encoding='utf-8') as f:\n      w=csv.DictWriter(f,fieldnames=list(rs[0].keys()))\n      w.writeheader();w.writerows(rs)\nsave_csv('trials_'+opt.label+'.csv',rows)\nsave_csv('sampled_trajectories_'+opt.label+'.csv',raw)\n(out/('config_'+opt.label+'.json')).write_text(json.dumps({\n   **ex,'label':opt.label,'checkpoint':str(opt.checkpoint),\n   'actual_dt_s':dt,'num_envs':N,'initial_state_sha256':initial_sha,\n   'root_and_joint_velocity_initial':'0',\n   'initial_joint_angles':'default + uniform(-0.02,0.02) rad',\n   'command_clamping':'no clipping applied; direct manual env.commands assignment',\n   'tracking_tolerance':'max(0.10 m/s, 15% of command)',\n   'steady_window_s':'[4, 18] for surviving trials',\n   'episode_done_interpreted_as':'termination/fall proxy'},\n   indent=2,ensure_ascii=False),encoding='utf-8')\n# Additional files for the movie. Original evaluation CSVs are written only to this NEW folder.\nassert capture_times, 'No live poses recorded; read the evaluation log.'\nnp.savez_compressed(out/('g1_pose_'+opt.label+'_1p9_rep1.npz'),\n    times=np.array(capture_times, dtype=np.float32),\n    body_xyz=np.stack(capture_xyz).astype(np.float32),\n    root_xyz=np.stack(capture_root).astype(np.float32),\n    yaw_rad=np.array(capture_yaw,dtype=np.float32),\n    start_root_xy=start_root_xy.astype(np.float32))\n(out/('g1_pose_info_'+opt.label+'.json')).write_text(json.dumps({\n    'model':opt.label,'command_mps':CAPTURE_SPEED,'rep':CAPTURE_REP,\n    'fps':CAPTURE_FPS,'dt_s':dt,'frames':len(capture_times),\n    'pose_is_rigid_body_centers':True,'body_names':body_names,\n    'initial_state_sha256':initial_sha,'recorded_until_s':float(capture_times[-1]),\n    'recorded_trial_ended':bool(capture_fallen),\n    'termination_time_s':float(fall_np[CAPTURE_ENV]) if capture_fallen else None,\n    'terminal_pose_recorded_before_reset':bool(capture_fallen and terminal_pose),\n    'source':'Isaac Gym physical states (not rendered camera images)'},indent=2,ensure_ascii=False),encoding='utf-8')\nprint('POSE_SAVED',opt.label,'frames',len(capture_times),flush=True)\nprint('saved',opt.label,'rows',len(rows),'survived',sum(x['survived_18s'] for x in rows),\n      'track',sum(x['survived_and_tracked'] for x in rows),'of',N,flush=True)\n", encoding='utf-8')
py_compile.compile(str(SIM_SCRIPT),doraise=True)
print('Capture script syntax OK:',SIM_SCRIPT)

## 4．1.9 m/s・試行1のP0・P1・P2骨格を記録（GPU）

高速歩行限界テストと同じ16環境（指令速度1.8，1.9 m/s×各8試行），シード2026を再現し，**1.9 m/sの試行1だけ**剛体位置を25 fpsで保存する．3方策の初期状態ハッシュとP1の終了時刻を照合する．


In [ ]:
import json,subprocess,shutil,time,os
from pathlib import Path
CONFIG_FILE=Path('/content/g1_video_1p9_experiment_config.json')
CONFIG_FILE.write_text(json.dumps({
 'speeds_mps':[1.8,1.9],
 'reps':8,'duration_s':18.0,'seed':2026,'sample_every_steps':5
}, indent=2),encoding='utf-8')
LOCAL=Path('/content/g1_video_1p9_models');LOCAL.mkdir(parents=True,exist_ok=True)
shas=[]
for label,source_path in MODELS.items():
    dest=LOCAL/(label+'.pt')
    shutil.copy2(source_path,dest)
    cmd=[str(PY38),str(SIM_SCRIPT),'--task=g1','--headless',
         '--sim_device=cuda:0','--rl_device=cuda:0',
         '--num_envs=16','--seed=2026','--checkpoint',str(dest),
         '--label',label,'--out',str(VIDEO_DIR),'--experiment-config',str(CONFIG_FILE)]
    log=VIDEO_DIR/f'video_capture_{label}.log'
    ts=time.time()
    print('Capture start:',label,flush=True)
    with log.open('w',encoding='utf-8') as fo:
        result=subprocess.run(cmd,cwd=str(repo),env=RUN_ENV,stdout=fo,stderr=subprocess.STDOUT)
    log_text=log.read_text(encoding='utf-8',errors='replace')
    print('Return:',result.returncode,'seconds:',round(time.time()-ts,1),'\n'+log_text[-2600:],flush=True)
    assert result.returncode==0, f'{label}: See {log}'
    info=json.loads((VIDEO_DIR/f'g1_pose_info_{label}.json').read_text(encoding='utf-8'))
    shas.append(info['initial_state_sha256'])
    assert info['frames'] >= 50, f'{label} has too few frames. Check {log}'
    if label in ('P0','P2'):
        assert info['frames'] >= 440, f'{label} ended early. Check {log}'
    if label=='P1':
        print('P1 rep1 termination:',info['termination_time_s'],'s')
        assert info['recorded_trial_ended'] and info['terminal_pose_recorded_before_reset'], 'P1 terminal pose missing'
assert len(set(shas))==1, '3方策の初期状態が異なります！'
REF_SHA='e21951aa383004fa45b91f2ed63b1150aefaf09627f7136fdd2161b8f5cd94e2'
print('3方策の同一初期状態 sha256:',shas[0])
if shas[0]!=REF_SHA:
    raise RuntimeError('直近の高速評価と初期状態SHAが一致しません．動画を比較に使う前に条件を確認してください．')
print('3方策の剛体座標記録が終了しました．')


## 5．同一世界座標の軌跡＋骨格を3画面MP4にする（CPU）

上段は3方策で縮尺を統一した世界XY軌跡，下段は実測剛体リンク中心を結んだ側面骨格．**P1終了後は最後の物理姿勢で静止し，P0・P2は18秒まで動く**．終了判定は完全な転倒・着地を必ずしも意味しない．


In [ ]:
from pathlib import Path
import sys,subprocess,py_compile
subprocess.run([sys.executable,'-m','pip','-q','install','Pillow','imageio','imageio-ffmpeg'],check=True)
RENDER_SCRIPT=Path('/content/g1_p012_1p9_fall_render.py')
RENDER_SCRIPT.write_text("import sys,json,csv,math\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFont\nimport imageio.v2 as imageio\n\nOUT=Path(sys.argv[1]);FPS=25\nPOLICIES=['P0','P1','P2']\nCOLORS={'P0':(39,110,205),'P1':(231,119,32),'P2':(211,50,64)}\nLEFT=(31,157,182);RIGHT=(214,135,73)\nW,H=1680,832\nPANEL_W=W//3\n\ntry:\n    base='/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'\n    bold='/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'\n    f12=ImageFont.truetype(base,15)\n    f15=ImageFont.truetype(base,18)\n    f24=ImageFont.truetype(bold,24)\n    f32=ImageFont.truetype(bold,31)\nexcept OSError:\n    f12=ImageFont.load_default();f15=f12;f24=f12;f32=f12\n\nD={};meta={}\nfor p in POLICIES:\n    D[p]=np.load(OUT/('g1_pose_'+p+'_1p9_rep1.npz'))\n    meta[p]=json.loads((OUT/('g1_pose_info_'+p+'.json')).read_text(encoding='utf-8'))\n    assert len(D[p]['times']) > 20, p\n    assert D[p]['body_xyz'].shape[0]==D[p]['root_xyz'].shape[0]\n    assert meta[p]['command_mps']==1.9 and meta[p]['rep']==1\n    assert meta[p]['fps']==FPS\nframes=max(len(D[p]['times']) for p in POLICIES)\nprint('Render frames:',frames, 'seconds',float(max(D[p]['times'][-1] for p in POLICIES)))\n\n# Use the SAME fixed map axes for all three policies. Local coordinates start at (0,0).\npaths={p:D[p]['root_xyz'][:,:2]-D[p]['start_root_xy'][None,:] for p in POLICIES}\nxall=np.hstack([paths[p][:,0] for p in POLICIES]+[np.array([0.])])\nyall=np.hstack([paths[p][:,1] for p in POLICIES]+[np.array([0.])])\nxmin=float(min(-1,xall.min()-1));xmax=float(max(28,xall.max()+1.0))\nymin=float(min(-5,yall.min()-1));ymax=float(max(5,yall.max()+1))\nscale=min(472/(xmax-xmin),277/(ymax-ymin))\n# Center plot to avoid different aspect distortion.\nxmid=(xmin+xmax)/2;ymid=(ymin+ymax)/2\nleft_labels=[]\n\n# Draw simplified bone chains from measured rigid-body LINK centers.\nfor p in POLICIES:\n    names=meta[p]['body_names'];idx={n:i for i,n in enumerate(names)}\n    chains=[]\n    for side,col in [('left',LEFT),('right',RIGHT)]:\n        leg=[x for x in ['pelvis',f'{side}_hip_yaw_link',f'{side}_hip_roll_link',f'{side}_hip_pitch_link',\n                         f'{side}_knee_link',f'{side}_ankle_pitch_link',f'{side}_ankle_roll_link'] if x in idx]\n        arm=[x for x in ['torso_link',f'{side}_shoulder_pitch_link',f'{side}_shoulder_roll_link',\n                         f'{side}_shoulder_yaw_link',f'{side}_elbow_link',f'{side}_wrist_roll_link'] if x in idx]\n        if len(leg)>=2:chains.append((leg,col))\n        if len(arm)>=2:chains.append((arm,col))\n    torso=[x for x in ['pelvis','waist_yaw_link','waist_roll_link','torso_link'] if x in idx]\n    if len(torso)>=2:chains.append((torso,(76,86,106)))\n    meta[p]['chains']=[([idx[n] for n in nodes],c) for nodes,c in chains]\n\n\ndef text(d,xy,s,fill=(41,52,70),font=f15):\n    d.text(xy,s,fill=fill,font=font)\n\ndef poly_arrow(draw,x,y,angle,color,r=19):\n    # Positive world yaw points positive Y (up on diagram).\n    ux,uy=math.cos(angle),-math.sin(angle)\n    x2=x+r*ux;y2=y+r*uy\n    draw.line((x,y,x2,y2),fill=color,width=5)\n    k=8\n    nx,ny=-uy,ux\n    draw.polygon([(x2+7*ux,y2+7*uy),(x2-8*ux+5*nx,y2-8*uy+5*ny),\n                  (x2-8*ux-5*nx,y2-8*uy-5*ny)],fill=color)\n\ndef draw_at(i):\n    t=float(D[POLICIES[0]]['times'][i])\n    im=Image.new('RGB',(W,H),(245,247,250));dr=ImageDraw.Draw(im)\n    dr.rectangle((0,0,W,76),fill=(24,36,54))\n    text(dr,(27,15),'UNITREE G1  /  1.9 m/s  /  Isaac Gym',fill=(255,255,255),font=f32)\n    text(dr,(W-290,24),f't = {t:5.2f} s   /   18 s',fill=(210,223,237),font=f15)\n    for col,p in enumerate(POLICIES):\n        px=col*PANEL_W\n        accent=COLORS[p]\n        dr.rounded_rectangle((px+8,82,px+PANEL_W-8,H-11),radius=14,fill=(255,255,255),outline=(219,227,236),width=2)\n        dr.rectangle((px+17,90,px+24,124),fill=accent)\n        text(dr,(px+34,91),f'{p}  |  contact tolerance '+{'P0':'0 ms','P1':'20 ms','P2':'40 ms'}[p],font=f24)\n        # Fixed-world XY path plot\n        box=(px+30,165,px+PANEL_W-30,450)\n        dr.rounded_rectangle(box,radius=9,fill=(249,251,253),outline=(228,232,238),width=2)\n        ox=px+PANEL_W/2;oy=304\n        def tr(x,y):return (int(ox+scale*(x-xmid)),int(oy-scale*(y-ymid)))\n        for gx in range(math.floor(xmin/5)*5,math.ceil(xmax/5)*5+1,5):\n            a=tr(gx,ymin);b=tr(gx,ymax)\n            dr.line((a,b),fill=(231,236,241),width=1)\n        for gy in range(math.floor(ymin/5)*5,math.ceil(ymax/5)*5+1,5):\n            a=tr(xmin,gy);b=tr(xmax,gy)\n            dr.line((a,b),fill=(231,236,241),width=1)\n        # Desired straight line y=0, initial command +X\n        a=tr(xmin,0);b=tr(xmax,0)\n        # dashed desired path\n        for s in range(a[0],b[0],18):dr.line((s,a[1],min(s+10,b[0]),a[1]),fill=(164,173,186),width=2)\n        j=min(i,len(D[p]['times'])-1)  # keep terminal pose after episode end\n        curve=paths[p][:j+1]\n        if len(curve)>1:dr.line([tr(float(x),float(y)) for x,y in curve],fill=accent,width=5,joint='curve')\n        last=curve[-1];cx,cy=tr(last[0],last[1])\n        dr.ellipse((cx-8,cy-8,cx+8,cy+8),fill=accent,outline=(255,255,255),width=2)\n        yaw=float(D[p]['yaw_rad'][j])\n        poly_arrow(dr,cx,cy,yaw,accent)\n        xval=float(last[0]);yval=float(last[1]);deg=math.degrees(yaw)\n        text(dr,(px+35,138),'TOP VIEW  |  World coordinates  |  grid 5 m',font=f15)\n        text(dr,(px+36,463),f'X = {xval:5.1f} m    Y = {yval:+5.1f} m    yaw = {deg:+6.1f} deg',font=f15)\n        # enlarged local X-Z skeleton showing gait motion, rotated into current heading\n        sx0=px+28;sy0=517;sx1=px+PANEL_W-28;sy1=764\n        dr.rounded_rectangle((sx0,sy0,sx1,sy1),radius=10,fill=(252,253,254),outline=(231,236,242),width=2)\n        text(dr,(px+40,497),'SIDE VIEW  |  Robot heading frame',font=f15)\n        ground=sy1-24\n        dr.line((sx0+12,ground,sx1-12,ground),fill=(166,173,184),width=2)\n        pos=D[p]['body_xyz'][j];root=D[p]['root_xyz'][j]\n        cos=math.cos(yaw);sin=math.sin(yaw)\n        def side_proj(pt):\n            dx=float(pt[0]-root[0]);dy=float(pt[1]-root[1]); fwd=cos*dx+sin*dy\n            return (int(px+PANEL_W/2+175*fwd),int(ground-175*float(pt[2])))\n        for js,color in meta[p]['chains']:\n            pts=[side_proj(pos[k]) for k in js]\n            dr.line(pts,fill=color,width=7,joint='curve')\n            for cx2,cy2 in pts:dr.ellipse((cx2-4,cy2-4,cx2+4,cy2+4),fill=color)\n        if not meta[p]['chains']:\n            for pt in pos:\n                a,b=side_proj(pt);dr.ellipse((a-3,b-3,a+3,b+3),fill=accent)\n        text(dr,(px+34,775),'Left leg: cyan   /   Right leg: ochre',font=f12)\n        if meta[p].get('recorded_trial_ended') and t >= float(meta[p]['recorded_until_s']):\n            end_t=float(meta[p]['termination_time_s'])\n            dr.rounded_rectangle((px+32,786,px+PANEL_W-32,814),radius=6,fill=(254,232,226))\n            text(dr,(px+42,789),f'EPISODE ENDED  t={end_t:.2f} s (final pose frozen)',fill=(130,46,35),font=f12)\n    return im\n\noutput=OUT/'P012_1p9mps_fall_rep1_3panel.mp4'\npreview=OUT/'P012_1p9mps_fall_rep1_preview.png'\nwith imageio.get_writer(str(output),fps=FPS,codec='libx264',quality=7,macro_block_size=16) as writer:\n    for i in range(frames):\n        frame=draw_at(i)\n        if i==min(frames-1,frames//2):frame.save(preview)\n        writer.append_data(np.asarray(frame))\n        if i%75==0:print('Rendered',i+1,'/',frames,flush=True)\n\n# One row per policy; these are a SINGLE representative trial, not 8-rep medians.\nrows=[]\nfor p in POLICIES:\n    last_idx=len(D[p]['times'])-1\n    last=paths[p][last_idx]\n    rows.append(dict(model=p,speed_command_mps=1.9,rep=1,\n        recorded_s=float(D[p]['times'][last_idx]),\n        delta_world_x_m=float(last[0]),delta_world_y_m=float(last[1]),\n        final_yaw_deg=float(math.degrees(float(D[p]['yaw_rad'][last_idx]))),\n        frames=len(D[p]['times']),\n        recorded_trial_ended=meta[p]['recorded_trial_ended'],\n        termination_time_s=meta[p].get('termination_time_s'))) \nwith (OUT/'P012_1p9mps_rep1_video_summary.csv').open('w',newline='',encoding='utf-8') as f:\n    w=csv.DictWriter(f,fieldnames=list(rows[0].keys()));w.writeheader();w.writerows(rows)\nprint('VIDEO_DONE:',output,'size MiB',round(output.stat().st_size/1024**2,1))\nprint('PREVIEW:',preview)\n", encoding='utf-8')
py_compile.compile(str(RENDER_SCRIPT),doraise=True)
result=subprocess.run([sys.executable,str(RENDER_SCRIPT),str(VIDEO_DIR)],text=True,capture_output=True)
print(result.stdout[-5500:]);print(result.stderr[-3000:])
assert result.returncode==0, '動画レンダリングに失敗．上のログを確認してください．'


## 6．Colabで完成動画を確認

In [ ]:
from IPython.display import display, Image, Video
from pathlib import Path
mp4=VIDEO_DIR/'P012_1p9mps_fall_rep1_3panel.mp4'
preview=VIDEO_DIR/'P012_1p9mps_fall_rep1_preview.png'
assert mp4.is_file(),mp4
print('MP4:',mp4,'MiB:',round(mp4.stat().st_size/1024**2,2))
if preview.exists():display(Image(filename=str(preview)))
# 再生用（大きすぎてColabが重くなる場合はこの行をスキップ）
display(Video(str(mp4), embed=True))
print('Google Drive内の保存先:',VIDEO_DIR)

## 結果の読み方

- 指令速度1.9 m/s，P0・P1・P2の**試行番号1**を同一初期状態で比較する．
- 上段の破線は世界X軸上の理想直進線，下段の骨格は剛体リンク中心の模式表示である．
- P1終了時はIsaac Gymが姿勢を初期化する直前の剛体位置を表示する．その後は姿勢を静止表示する．**終了後の動きは測定されていない**．
- エピソード終了は，接触・roll・pitchなどに基づく終了判定であり，床面への完全な着地とは限らない．
- **これは1試行の例示**であり，非転倒率の統計的要約ではない．非転倒率は別ノートブック`G1_P0_P1_P2_High_Speed_Fall_Limit2.ipynb`で評価する．
- 保存先`video_1p9_fall_rep1`は新規であり，1.4 m/s用`video_1p4_compare`と高速評価`fine_1p9_20261028`を上書きしない．
